In [4]:
# C1 — Question Type Performance Breakdown (SLAKE specific)

import os
import json
import glob
import pandas as pd
from datasets import load_dataset

def normalize_word(word):
    """Basic unigram normalization for Token F1."""
    return str(word).lower().strip(",.?\"'!")

def compute_token_f1(pred, ground_truth):
    """Compute exact Token F1 offline without loading evaluation libraries."""
    pred_tokens = [normalize_word(w) for w in str(pred).split()]
    gt_tokens = [normalize_word(w) for w in str(ground_truth).split()]
    
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
        
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gt_tokens)
    return 2 * (prec * rec) / (prec + rec)

def run_slake_content_type_analysis_to_csv():
    print("Loading original SLAKE dataset to map content_types...")
    # This reads from your local HuggingFace cache
    slake_hf = load_dataset('BoKelvin/SLAKE', split='test')
    slake_en = [s for s in slake_hf if s['q_lang'] == 'en']
    
    # Map the zero-based inference index (idx) to the dataset's content_type
    idx_to_content_type = {i: sample['content_type'] for i, sample in enumerate(slake_en)}
    
    # Point to the directory containing your judged jsonl files
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    slake_files = glob.glob(os.path.join(judged_dir, '*slake*.jsonl'))
    
    records = []
    for filepath in slake_files:
        filename = os.path.basename(filepath)
        # Extract the model name from the file string
        model_name = filename.split('__')[0].replace('_', '/')
        
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                idx = data.get('idx')
                
                # Ensure the index exists and maps correctly to the English split
                if idx is None or idx not in idx_to_content_type:
                    continue
                    
                records.append({
                    'Model': model_name,
                    'Content Type': idx_to_content_type[idx],
                    'Token F1': compute_token_f1(data.get('prediction', ''), data.get('ground_truth', '')) * 100.0,
                    'Judge Acc': 100.0 if data.get('judge_score', 0) >= 4 else 0.0
                })
                
    df = pd.DataFrame(records)
    
    # Extract the absolute sample count for each content_type
    support = df[df['Model'] == df['Model'].unique()[0]]['Content Type'].value_counts()
    
    # Generate F1 Pivot Table
    f1_pivot = df.pivot_table(index='Content Type', columns='Model', values='Token F1', aggfunc='mean').round(2)
    f1_pivot.insert(0, 'Count', support)
    f1_pivot = f1_pivot.sort_values(by='Count', ascending=False)
    
    # Generate Judge Accuracy Pivot Table
    judge_pivot = df.pivot_table(index='Content Type', columns='Model', values='Judge Acc', aggfunc='mean').round(2)
    judge_pivot.insert(0, 'Count', support)
    judge_pivot = judge_pivot.sort_values(by='Count', ascending=False)
    
    # --- DIRECTORY AND CSV EXPORT LOGIC ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    f1_csv_path = os.path.join(output_dir, 'slake_content_type_f1.csv')
    judge_csv_path = os.path.join(output_dir, 'slake_content_type_judge.csv')
    
    f1_pivot.to_csv(f1_csv_path)
    judge_pivot.to_csv(judge_csv_path)
    
    print(f"\n✅ Successfully saved F1 analysis to: {f1_csv_path}")
    print(f"✅ Successfully saved Judge analysis to: {judge_csv_path}")
    
    return f1_pivot, judge_pivot

# Execute the standalone analysis
df_f1, df_judge = run_slake_content_type_analysis_to_csv()

Loading original SLAKE dataset to map content_types...

✅ Successfully saved F1 analysis to: /Users/shriyanshraj/vlm_benchmark/outputs/error_analysis/slake_content_type_f1.csv
✅ Successfully saved Judge analysis to: /Users/shriyanshraj/vlm_benchmark/outputs/error_analysis/slake_content_type_judge.csv


In [6]:
# B3 — Closed vs Open Performance Gap

import os
import json
import glob
import pandas as pd
import numpy as np

def run_closed_open_gap_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    records = []
    print("Parsing LLM Judge results to compute the Closed-Open semantic gap...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        
        # Cleanly extract the model name
        parts = filename.split('__')
        model_name = parts[0].replace('_', '/')
        
        # Standardize the dataset names for a clean matrix
        dataset_raw = parts[1].lower() if len(parts) > 1 else 'unknown'
        if 'slake' in dataset_raw: dataset_name = 'SLAKE'
        elif 'vqa_rad' in dataset_raw: dataset_name = 'VQA-RAD'
        elif 'vqav2' in dataset_raw: dataset_name = 'VQAv2'
        elif 'okvqa' in dataset_raw: dataset_name = 'OK-VQA'
        else: dataset_name = dataset_raw
            
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                is_closed = data.get('is_closed', False)
                judge_score = data.get('judge_score')
                
                # --- THE FIX: Skip records where the judge failed (returned null) ---
                if judge_score is None:
                    continue
                # --------------------------------------------------------------------
                    
                # Semantic correctness based on judge
                is_correct = 1 if judge_score >= 4 else 0
                
                records.append({
                    'Model': model_name,
                    'Dataset': dataset_name,
                    'Question Type': 'Closed' if is_closed else 'Open',
                    'Is Correct': is_correct
                })
                
    df = pd.DataFrame(records)
    
    # Calculate win rates (accuracies) by Model, Dataset, and Question Type
    agg_df = df.groupby(['Model', 'Dataset', 'Question Type'])['Is Correct'].mean().unstack() * 100.0
    
    # Calculate the Gap (Closed - Open)
    if 'Closed' in agg_df.columns and 'Open' in agg_df.columns:
        agg_df['Performance Gap'] = agg_df['Closed'] - agg_df['Open']
    else:
        print("Warning: Missing either closed or open data across datasets.")
        return None
        
    # Pivot the table specifically for the Gap metric
    gap_pivot = agg_df.reset_index().pivot(index='Model', columns='Dataset', values='Performance Gap').round(2)
    
    # Ensure columns are in a logical order if they exist
    col_order = [c for c in ['SLAKE', 'VQA-RAD', 'VQAv2', 'OK-VQA'] if c in gap_pivot.columns]
    gap_pivot = gap_pivot[col_order]
    
    # --- DIRECTORY AND CSV EXPORT LOGIC ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    gap_csv_path = os.path.join(output_dir, 'closed_vs_open_gap.csv')
    gap_pivot.to_csv(gap_csv_path)
    
    print("\n=== Closed vs. Open Accuracy Gap (Judge Semantic Scoring) ===")
    print("A smaller number means the model is equally good at generating answers as it is at verifying them.\n")
    print(gap_pivot.to_string())
    
    print(f"\n✅ Successfully saved Gap analysis to: {gap_csv_path}")
    
    return gap_pivot

# Execute the standalone analysis
df_gap = run_closed_open_gap_analysis()

Parsing LLM Judge results to compute the Closed-Open semantic gap...

=== Closed vs. Open Accuracy Gap (Judge Semantic Scoring) ===
A smaller number means the model is equally good at generating answers as it is at verifying them.

Dataset                                            SLAKE  VQA-RAD  VQAv2  OK-VQA
Model                                                                           
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL  17.52    31.39    NaN     NaN
google/gemma-3-4b-it                               22.79    24.97   3.40     NaN
google/medgemma-4b-it                              16.37    21.31    NaN     NaN
llava-hf/llava-v1.6-mistral-7b-hf                  21.52    29.67   3.46     NaN
microsoft/llava-med-v1.5-mistral-7b                 7.15    -2.08    NaN     NaN

✅ Successfully saved Gap analysis to: /Users/shriyanshraj/vlm_benchmark/outputs/error_analysis/closed_vs_open_gap.csv


In [7]:
# A1 — Modality Confusion (medical datasets only) and A2 — Anatomical Hallucination (medical datasets only)

import os
import json
import glob
import re
import pandas as pd
from datasets import load_dataset

def normalize_text(text):
    """Normalize text for basic token matching."""
    return str(text).lower().strip(",.?\"'!\n ")

def compute_token_f1(pred, gt):
    """Calculate exact Token F1 offline."""
    pred_tokens = [normalize_text(w) for w in str(pred).split()]
    gt_tokens = [normalize_text(w) for w in str(gt).split()]
    
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
        
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gt_tokens)
    return 2 * (prec * rec) / (prec + rec)

def run_clinical_error_analysis():
    print("Loading original SLAKE dataset to map content_types...")
    slake_hf = load_dataset('BoKelvin/SLAKE', split='test')
    slake_en = [s for s in slake_hf if s['q_lang'] == 'en']
    slake_idx_to_type = {i: sample['content_type'] for i, sample in enumerate(slake_en)}
    
    # Define our clinical vocabularies
    modality_keywords = ['ct', 'mri', 'x-ray', 'xray', 'ultrasound', 'us', 'pet', 'radiograph']
    anatomy_vocab = [
        'lung', 'heart', 'liver', 'spleen', 'kidney', 'brain', 'chest', 
        'abdomen', 'pelvis', 'spine', 'neck', 'thyroid', 'pancreas', 
        'gallbladder', 'bladder', 'colon', 'stomach'
    ]
    
    # Setup directories
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    # We only want to analyze the medical datasets for this clinical evaluation
    med_files = glob.glob(os.path.join(judged_dir, '*slake*.jsonl')) + \
                glob.glob(os.path.join(judged_dir, '*vqa_rad*.jsonl'))
                
    # Trackers for A1 (Modality) and A2 (Anatomy)
    modality_stats = {}
    anatomy_stats = {}
    
    print("Scanning medical JSONL outputs for clinical errors...")
    for filepath in med_files:
        filename = os.path.basename(filepath)
        model_name = filename.split('__')[0].replace('_', '/')
        
        if model_name not in modality_stats:
            modality_stats[model_name] = {'Total Modality Qs': 0, 'Modality Confusions': 0}
            anatomy_stats[model_name] = {'Total Zero-F1 Open Qs': 0, 'Anatomical Hallucinations': 0}
            
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                idx = data.get('idx')
                dataset = data.get('dataset', '').lower()
                question = normalize_text(data.get('question', ''))
                gt = normalize_text(data.get('ground_truth', ''))
                pred = normalize_text(data.get('prediction', ''))
                is_closed = data.get('is_closed', False)
                
                # --- A1: Modality Confusion Logic ---
                is_modality_q = False
                if 'slake' in dataset and idx in slake_idx_to_type:
                    is_modality_q = (slake_idx_to_type[idx] == 'Modality')
                elif 'modality' in question:
                    is_modality_q = True
                
                # Fallback: if the GT is literally just a modality, it's a modality question
                if any(m == gt for m in modality_keywords):
                    is_modality_q = True
                    
                if is_modality_q:
                    modality_stats[model_name]['Total Modality Qs'] += 1
                    
                    # Check if the prediction contains ANY modality keyword
                    pred_has_modality = any(re.search(rf'\b{m}\b', pred) for m in modality_keywords)
                    # Check if the ground truth contains that modality keyword
                    pred_matches_gt = any(re.search(rf'\b{m}\b', pred) for m in modality_keywords if m in gt)
                    
                    # A confusion happens when the model names a modality, but it's the WRONG modality
                    if pred_has_modality and not pred_matches_gt:
                        modality_stats[model_name]['Modality Confusions'] += 1
                        
                # --- A2: Anatomical Hallucination Logic ---
                if not is_closed:
                    f1 = compute_token_f1(pred, gt)
                    if f1 == 0.0:
                        anatomy_stats[model_name]['Total Zero-F1 Open Qs'] += 1
                        
                        # Find all anatomy words in prediction and ground truth
                        pred_organs = [org for org in anatomy_vocab if re.search(rf'\b{org}\b', pred)]
                        gt_organs = [org for org in anatomy_vocab if re.search(rf'\b{org}\b', gt)]
                        
                        # If the prediction names an organ that is NOT in the ground truth
                        hallucinated_organs = [org for org in pred_organs if org not in gt_organs]
                        
                        if len(hallucinated_organs) > 0:
                            anatomy_stats[model_name]['Anatomical Hallucinations'] += 1

    # Format DataFrames
    df_mod = pd.DataFrame.from_dict(modality_stats, orient='index')
    df_mod['Confusion Rate (%)'] = (df_mod['Modality Confusions'] / df_mod['Total Modality Qs'] * 100).round(2)
    df_mod = df_mod.sort_values(by='Confusion Rate (%)', ascending=True)
    
    df_anat = pd.DataFrame.from_dict(anatomy_stats, orient='index')
    df_anat['Hallucination Rate (%)'] = (df_anat['Anatomical Hallucinations'] / df_anat['Total Zero-F1 Open Qs'] * 100).round(2)
    df_anat = df_anat.sort_values(by='Hallucination Rate (%)', ascending=True)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    mod_csv = os.path.join(output_dir, 'modality_confusion_rates.csv')
    anat_csv = os.path.join(output_dir, 'anatomical_hallucination_rates.csv')
    
    df_mod.to_csv(mod_csv)
    df_anat.to_csv(anat_csv)
    
    print("\n=== A1: Modality Confusion Rates ===")
    print("How often a model confidently hallucinates the WRONG imaging modality.")
    print(df_mod.to_string())
    
    print("\n=== A2: Anatomical Hallucination Rates ===")
    print("On completely failed open-ended questions (F1=0), how often did the model confidently name a completely wrong organ?")
    print(df_anat.to_string())

    print(f"\n✅ Successfully saved A1 analysis to: {mod_csv}")
    print(f"✅ Successfully saved A2 analysis to: {anat_csv}")
    
    return df_mod, df_anat

# Execute the standalone clinical analysis
df_modality, df_anatomy = run_clinical_error_analysis()

Loading original SLAKE dataset to map content_types...
Scanning medical JSONL outputs for clinical errors...

=== A1: Modality Confusion Rates ===
How often a model confidently hallucinates the WRONG imaging modality.
                                                   Total Modality Qs  Modality Confusions  Confusion Rate (%)
google/medgemma-4b-it                                            124                    5                4.03
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL                124                    6                4.84
google/gemma-3-4b-it                                             124                    6                4.84
microsoft/llava-med-v1.5-mistral-7b                              124                   11                8.87
llava-hf/llava-v1.6-mistral-7b-hf                                124                   37               29.84

=== A2: Anatomical Hallucination Rates ===
On completely failed open-ended questions (F1=0), how often did the model conf

In [9]:
# B4 — Answer Length Analysis

import os
import json
import glob
import pandas as pd

def normalize_text(text):
    """Normalize text for basic token matching."""
    return str(text).lower().strip(",.?\"'!\n ")

def compute_token_f1(pred, gt):
    """Calculate exact Token F1 offline."""
    pred_tokens = [normalize_text(w) for w in str(pred).split()]
    gt_tokens = [normalize_text(w) for w in str(gt).split()]
    
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
        
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gt_tokens)
    return 2 * (prec * rec) / (prec + rec)

def run_length_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    records = []
    print("Scanning JSONL outputs to analyze answer verbosity vs. correctness...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        model_name = filename.split('__')[0].replace('_', '/')
        
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                
                # Get the raw output (if it doesn't exist, fallback to prediction)
                raw_output = data.get('raw_output', data.get('prediction', ''))
                
                # Calculate word length
                word_count = len(str(raw_output).split())
                
                # Bin the lengths
                if word_count < 20:
                    length_bin = 'Short (<20)'
                elif word_count <= 100:
                    length_bin = 'Medium (20-100)'
                else:
                    length_bin = 'Long (>100)'
                    
                # Calculate metrics
                f1 = compute_token_f1(data.get('prediction', ''), data.get('ground_truth', '')) * 100.0
                judge_score = data.get('judge_score')
                
                # Skip unjudged records
                if judge_score is None:
                    continue
                    
                judge_acc = 100.0 if judge_score >= 4 else 0.0
                
                records.append({
                    'Model': model_name,
                    'Length Bin': length_bin,
                    'F1': f1,
                    'Judge Acc': judge_acc
                })
                
    df = pd.DataFrame(records)
    
    # Enforce categorical order so the table rows display logically
    bin_order = ['Short (<20)', 'Medium (20-100)', 'Long (>100)']
    df['Length Bin'] = pd.Categorical(df['Length Bin'], categories=bin_order, ordered=True)
    
    # Group by Model and Length Bin, computing the mean
    agg_df = df.groupby(['Model', 'Length Bin'], observed=False).agg(
        Count=('F1', 'count'),
        Avg_F1=('F1', 'mean'),
        Avg_Judge_Acc=('Judge Acc', 'mean')
    ).round(2)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'answer_length_analysis.csv')
    agg_df.to_csv(csv_path)
    
    print("\n=== B4: Answer Length Analysis (Verbosity vs. Correctness) ===")
    print("Does generating more text lead to deeper reasoning, or hallucination drift?")
    print(agg_df.to_string())
    
    print(f"\n✅ Successfully saved Answer Length analysis to: {csv_path}")
    
    return agg_df

# Execute the standalone analysis
df_length = run_length_analysis()

Scanning JSONL outputs to analyze answer verbosity vs. correctness...

=== B4: Answer Length Analysis (Verbosity vs. Correctness) ===
Does generating more text lead to deeper reasoning, or hallucination drift?
                                                                   Count  Avg_F1  Avg_Judge_Acc
Model                                             Length Bin                                   
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL Short (<20)        792   59.10          69.95
                                                  Medium (20-100)    587   43.28          55.54
                                                  Long (>100)        132   32.81          46.21
google/gemma-3-4b-it                              Short (<20)       1118   55.12          60.64
                                                  Medium (20-100)   2174   36.33          48.71
                                                  Long (>100)        198    9.86          19.19
google/medgemma-4b-it 

In [10]:
# C3 — Judge Score Distribution Analysis

import os
import json
import glob
import pandas as pd

def run_judge_distribution_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    records = []
    print("Parsing Judge scores to isolate structural distribution matrices...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        model_name = filename.split('__')[0].replace('_', '/')
        
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                judge_score = data.get('judge_score')
                
                # Filter out failures or null entries
                if judge_score is None:
                    continue
                
                try:
                    score_int = int(judge_score)
                    if score_int in [1, 2, 3, 4, 5]:
                        records.append({
                            'Model': model_name,
                            'Score': f"Score {score_int}"
                        })
                except (ValueError, TypeError):
                    continue
                    
    df = pd.DataFrame(records)
    
    # Calculate cross-tabulations (counts)
    dist_counts = pd.crosstab(df['Model'], df['Score'])
    
    # Enforce standard column ordering for the 1-5 scale
    score_cols = [f"Score {i}" for i in range(1, 6)]
    dist_counts = dist_counts.reindex(columns=score_cols, fill_value=0)
    
    # Calculate row totals to serve as our denominators
    total_valid_samples = dist_counts.sum(axis=1)
    
    # Divide counts by totals to generate clean percentage footprints
    dist_pct = dist_counts.div(total_valid_samples, axis=0) * 100.0
    dist_pct.insert(0, 'Total Sample Count', total_valid_samples)
    dist_pct = dist_pct.round(2)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'judge_score_distribution.csv')
    dist_pct.to_csv(csv_path)
    
    print("\n=== C3: LLM Judge Score Distribution Analysis (Percentages) ===")
    print("Reveals the exact behavioral footprint and risk profile of each model architecture.\n")
    print(dist_pct.to_string())
    
    print(f"\n✅ Successfully saved Score Distribution analysis to: {csv_path}")
    
    return dist_pct

# Execute the standalone analysis
df_dist = run_judge_distribution_analysis()

Parsing Judge scores to isolate structural distribution matrices...

=== C3: LLM Judge Score Distribution Analysis (Percentages) ===
Reveals the exact behavioral footprint and risk profile of each model architecture.

Score                                              Total Sample Count  Score 1  Score 2  Score 3  Score 4  Score 5
Model                                                                                                             
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL                1511    21.77     7.88     8.07    18.20    44.08
google/gemma-3-4b-it                                             3490    32.38    11.09     5.67    12.52    38.34
google/medgemma-4b-it                                            1512    15.01     6.15     8.07    15.61    55.16
llava-hf/llava-v1.6-mistral-7b-hf                                3488    29.59     8.29     4.82    11.55    45.76
microsoft/llava-med-v1.5-mistral-7b                              1512    32.47     5.22    1

In [12]:
# B5 — Cross-Domain Degradation Profile

import os
import json
import glob
import pandas as pd

def run_domain_penalty_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    # We only care about the models that ran on BOTH general and medical data
    target_models = ['google/gemma-3-4b-it', 'llava-hf/llava-v1.6-mistral-7b-hf']
    records = []
    
    print("Calculating the Medical Domain Transfer Penalty...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        model_name = filename.split('__')[0].replace('_', '/')
        
        if model_name not in target_models:
            continue
            
        # Standardize dataset names
        dataset_raw = filename.split('__')[1].lower() if len(filename.split('__')) > 1 else 'unknown'
        if 'slake' in dataset_raw: dataset_name = 'SLAKE'
        elif 'vqa_rad' in dataset_raw: dataset_name = 'VQA-RAD'
        elif 'vqav2' in dataset_raw: dataset_name = 'VQAv2'
        else: continue # Skip OK-VQA for this specific baseline comparison
            
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                judge_score = data.get('judge_score')
                
                if judge_score is None:
                    continue
                    
                is_correct = 100.0 if judge_score >= 4 else 0.0
                
                records.append({
                    'Model': model_name,
                    'Dataset': dataset_name,
                    'Judge Acc': is_correct
                })
                
    df = pd.DataFrame(records)
    
    # Calculate overall accuracy per model per dataset
    acc_df = df.groupby(['Model', 'Dataset'])['Judge Acc'].mean().unstack().round(2)
    
    # Ensure all required columns exist
    for col in ['VQAv2', 'SLAKE', 'VQA-RAD']:
        if col not in acc_df.columns:
            acc_df[col] = pd.NA
            
    # Calculate the exact domain penalties
    acc_df['SLAKE Penalty'] = acc_df['VQAv2'] - acc_df['SLAKE']
    acc_df['VQA-RAD Penalty'] = acc_df['VQAv2'] - acc_df['VQA-RAD']
    
    # Reorder columns for a clean logical flow
    col_order = ['VQAv2', 'SLAKE', 'SLAKE Penalty', 'VQA-RAD', 'VQA-RAD Penalty']
    final_df = acc_df[col_order]
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'medical_domain_penalty.csv')
    final_df.to_csv(csv_path)
    
    print("\n=== C5: Medical Domain Penalty Analysis ===")
    print("Quantifying how much performance degrades when moving from general images to medical scans.\n")
    print(final_df.to_string())
    
    print(f"\n✅ Successfully saved Domain Penalty analysis to: {csv_path}")
    
    return final_df

# Execute the standalone analysis
df_penalty = run_domain_penalty_analysis()

Calculating the Medical Domain Transfer Penalty...

=== C5: Medical Domain Penalty Analysis ===
Quantifying how much performance degrades when moving from general images to medical scans.

Dataset                            VQAv2  SLAKE  SLAKE Penalty  VQA-RAD  VQA-RAD Penalty
Model                                                                                   
google/gemma-3-4b-it               58.30  55.14           3.16    45.90            12.40
llava-hf/llava-v1.6-mistral-7b-hf  70.22  50.14          20.08    45.01            25.21

✅ Successfully saved Domain Penalty analysis to: /Users/shriyanshraj/vlm_benchmark/outputs/error_analysis/medical_domain_penalty.csv


In [1]:
# A4 — Conversational Drift

import os
import json
import glob
import pandas as pd

def run_conversational_drift_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    stats = []
    print("Scanning raw outputs to detect Conversational Drift and Extraction Leakage...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        
        # Cleanly extract model and dataset names
        parts = filename.split('__')
        model_name = parts[0].replace('_', '/')
        dataset_raw = parts[1].lower() if len(parts) > 1 else 'unknown'
        
        if 'slake' in dataset_raw: dataset_name = 'SLAKE'
        elif 'vqa_rad' in dataset_raw: dataset_name = 'VQA-RAD'
        elif 'vqav2' in dataset_raw: dataset_name = 'VQAv2'
        elif 'okvqa' in dataset_raw: dataset_name = 'OK-VQA'
        else: dataset_name = dataset_raw
            
        total_qs = 0
        leakage_count = 0
        drift_count = 0
        
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                
                # Get the raw output and the extracted prediction
                raw_output = str(data.get('raw_output', data.get('prediction', '')))
                prediction = str(data.get('prediction', ''))
                
                total_qs += 1
                
                # Check for the structural anchor
                has_anchor = 'final answer:' in raw_output.lower()
                
                # 1. Extraction Leakage
                # The model generated the anchor, but the extractor pulled a blank string OR pulled a massive paragraph (> 5 words)
                pred_word_count = len(prediction.split())
                if has_anchor and (pred_word_count == 0 or pred_word_count > 5):
                    leakage_count += 1
                    
                # 2. Conversational Drift
                # The model completely ignored the structural prompt, never generated the anchor, and rambled for > 200 characters
                if not has_anchor and len(raw_output) > 200:
                    drift_count += 1
                    
        stats.append({
            'Model': model_name,
            'Dataset': dataset_name,
            'Total Qs': total_qs,
            'Leakage Count': leakage_count,
            'Leakage Rate (%)': round((leakage_count / total_qs) * 100, 2) if total_qs > 0 else 0,
            'Drift Count': drift_count,
            'Drift Rate (%)': round((drift_count / total_qs) * 100, 2) if total_qs > 0 else 0
        })
        
    df = pd.DataFrame(stats)
    
    # Sort for readability: Group by Model, then by Dataset
    df = df.sort_values(by=['Model', 'Dataset']).reset_index(drop=True)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'conversational_drift_analysis.csv')
    df.to_csv(csv_path, index=False)
    
    print("\n=== A4: Conversational Drift & Extraction Leakage ===")
    print("Leakage: Model used 'Final Answer:' but broke extraction (e.g. buried it in text).")
    print("Drift: Model completely ignored prompt constraints and rambled (>200 chars).\n")
    print(df.to_string(index=False))
    
    print(f"\n✅ Successfully saved Drift analysis to: {csv_path}")
    
    return df

# Execute the standalone analysis
df_drift = run_conversational_drift_analysis()

Scanning raw outputs to detect Conversational Drift and Extraction Leakage...

=== A4: Conversational Drift & Extraction Leakage ===
Leakage: Model used 'Final Answer:' but broke extraction (e.g. buried it in text).
Drift: Model completely ignored prompt constraints and rambled (>200 chars).

                                            Model Dataset  Total Qs  Leakage Count  Leakage Rate (%)  Drift Count  Drift Rate (%)
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL   SLAKE      1061              2              0.19           27            2.54
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL VQA-RAD       451              0              0.00            0            0.00
                             google/gemma-3-4b-it  OK-VQA      1000              1              0.10          334           33.40
                             google/gemma-3-4b-it   SLAKE      1061              1              0.09           84            7.92
                             google/gemma-3-4b-it VQA-RA

In [4]:
# A3 — Spatial Reasoning Failure

import os
import json
import glob
import pandas as pd

def normalize_text(text):
    """Normalize text for basic token matching."""
    return str(text).lower().strip(",.?\"'!\n ")

def compute_token_f1(pred, gt):
    """Calculate exact Token F1 offline."""
    pred_tokens = [normalize_text(w) for w in str(pred).split()]
    gt_tokens = [normalize_text(w) for w in str(gt).split()]
    
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
        
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gt_tokens)
    return 2 * (prec * rec) / (prec + rec)

def run_spatial_failure_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    # Define spatial anchoring keywords
    spatial_keywords = ["where", "location", "left", "right", "largest", "biggest", "how many", "count", "side"]
    
    records = []
    print("Extracting spatial reasoning cohorts across all evaluation layers...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        
        # Cleanly extract model and dataset labels
        parts = filename.split('__')
        model_name = parts[0].replace('_', '/')
        dataset_raw = parts[1].lower() if len(parts) > 1 else 'unknown'
        
        if 'slake' in dataset_raw: 
            dataset_name = 'SLAKE'
            domain = 'Medical'
        elif 'vqa_rad' in dataset_raw: 
            dataset_name = 'VQA-RAD'
            domain = 'Medical'
        elif 'vqav2' in dataset_raw: 
            dataset_name = 'VQAv2'
            domain = 'General'
        elif 'okvqa' in dataset_raw: 
            dataset_name = 'OK-VQA'
            domain = 'General'
        else: 
            continue
            
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                question = str(data.get('question', '')).lower()
                pred = str(data.get('prediction', '')).strip()
                gt = str(data.get('ground_truth', '')).strip()
                
                # Condition 1: Check if the prompt falls into the spatial reasoning taxonomy
                is_spatial_q = any(keyword in question for keyword in spatial_keywords)
                
                if is_spatial_q:
                    f1 = compute_token_f1(pred, gt)
                    
                    # Condition 2: Spatial failure occurs when the model responds (non-empty) but hits an absolute F1 zero
                    is_spatial_failure = 1 if (f1 == 0.0 and pred != "") else 0
                    
                    records.append({
                        'Model': model_name,
                        'Dataset': dataset_name,
                        'Domain': domain,
                        'Is Spatial Failure': is_spatial_failure
                    })
                    
    df = pd.DataFrame(records)
    
    # Aggregate to find total spatial queries and absolute failure counts
    summary = df.groupby(['Model', 'Domain', 'Dataset']).agg(
        Spatial_Qs=('Is Spatial Failure', 'count'),
        Spatial_Failures=('Is Spatial Failure', 'sum')
    ).reset_index()
    
    # --- THE FIX: Using the exact column name with the underscore ---
    summary['Spatial Failure Rate (%)'] = (summary['Spatial_Failures'] / summary['Spatial_Qs'] * 100).round(2)
    # ----------------------------------------------------------------
    
    # Sort logically by Model, then Domain type
    summary = summary.sort_values(by=['Model', 'Domain'], ascending=[True, False]).reset_index(drop=True)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'spatial_reasoning_failures.csv')
    summary.to_csv(csv_path, index=False)
    
    print("\n=== A8: Spatial Reasoning Failure Analysis ===")
    print("Measures the percentage of spatial questions where a model confidently generated a completely wrong coordinate/direction.\n")
    print(summary.to_string(index=False))
    
    print(f"\n✅ Successfully saved Spatial analysis to: {csv_path}")
    
    return summary

# Execute the standalone analysis
df_spatial = run_spatial_failure_analysis()

Extracting spatial reasoning cohorts across all evaluation layers...

=== A8: Spatial Reasoning Failure Analysis ===
Measures the percentage of spatial questions where a model confidently generated a completely wrong coordinate/direction.

                                            Model  Domain Dataset  Spatial_Qs  Spatial_Failures  Spatial Failure Rate (%)
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL Medical   SLAKE         305               212                     69.51
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL Medical VQA-RAD          82                48                     58.54
                             google/gemma-3-4b-it Medical   SLAKE         305               224                     73.44
                             google/gemma-3-4b-it Medical VQA-RAD          82                50                     60.98
                             google/gemma-3-4b-it General  OK-VQA          95                78                     82.11
                            

In [5]:
# A5 — CoT-Induced Hallucination

import os
import json
import glob
import re
import pandas as pd

def normalize_text(text):
    """Normalize text for basic token matching."""
    return str(text).lower().strip(",.?\"'!\n ")

def compute_token_f1(pred, gt):
    """Calculate exact Token F1 offline."""
    pred_tokens = [normalize_text(w) for w in str(pred).split()]
    gt_tokens = [normalize_text(w) for w in str(gt).split()]
    
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
        
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gt_tokens)
    return 2 * (prec * rec) / (prec + rec)

def run_cot_hallucination_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    judged_files = glob.glob(os.path.join(judged_dir, '*_judged.jsonl'))
    
    # Standard English stopwords to prevent false positive matches on words like "the" or "on"
    stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'it', 'this', 'that', 
                 'of', 'in', 'on', 'at', 'to', 'and', 'or', 'with', 'as', 'by', 'image', 'scan'}
    
    records = []
    print("Parsing raw outputs to detect Chain-of-Thought (CoT) induced hallucinations...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        
        parts = filename.split('__')
        model_name = parts[0].replace('_', '/')
        dataset_raw = parts[1].lower() if len(parts) > 1 else 'unknown'
        
        if 'slake' in dataset_raw: dataset_name = 'SLAKE'
        elif 'vqa_rad' in dataset_raw: dataset_name = 'VQA-RAD'
        elif 'vqav2' in dataset_raw: dataset_name = 'VQAv2'
        elif 'okvqa' in dataset_raw: dataset_name = 'OK-VQA'
        else: continue
            
        failed_cot_count = 0
        cot_hallucination_count = 0
        
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                raw_output = str(data.get('raw_output', data.get('prediction', '')))
                pred = str(data.get('prediction', ''))
                gt = str(data.get('ground_truth', ''))
                
                f1 = compute_token_f1(pred, gt)
                
                # We only care about complete failures (F1=0) where the model attempted the CoT format
                match = re.search(r'(?i)final\s+answer\s*:', raw_output)
                if match and f1 == 0.0:
                    failed_cot_count += 1
                    
                    # Extract the reasoning block (everything before the anchor)
                    reasoning_block = raw_output[:match.start()]
                    reasoning_norm = normalize_text(reasoning_block)
                    reasoning_tokens = set(reasoning_norm.split())
                    
                    gt_norm = normalize_text(gt)
                    gt_tokens = set([w for w in gt_norm.split() if w not in stopwords])
                    
                    has_gt_in_reasoning = False
                    
                    # Check 1: Does the exact GT string appear in the reasoning?
                    if gt_norm in reasoning_norm and len(gt_norm) > 0:
                        has_gt_in_reasoning = True
                    # Check 2: Does any significant token from the GT appear in the reasoning?
                    elif len(gt_tokens & reasoning_tokens) > 0:
                        has_gt_in_reasoning = True
                        
                    if has_gt_in_reasoning:
                        cot_hallucination_count += 1
                        
        records.append({
            'Model': model_name,
            'Dataset': dataset_name,
            'Failed CoT Qs': failed_cot_count,
            'CoT Hallucinations': cot_hallucination_count,
            'CoT Hallucination Rate (%)': round((cot_hallucination_count / failed_cot_count) * 100, 2) if failed_cot_count > 0 else 0.0
        })
        
    df = pd.DataFrame(records)
    
    # Filter out empty records and sort
    df = df[df['Failed CoT Qs'] > 0]
    df = df.sort_values(by=['Model', 'Dataset']).reset_index(drop=True)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'cot_induced_hallucinations.csv')
    df.to_csv(csv_path, index=False)
    
    print("\n=== A5: CoT-Induced Hallucination Analysis ===")
    print("Measures how often the model generated the correct answer during reasoning, but outputted the wrong final answer.\n")
    print(df.to_string(index=False))
    
    print(f"\n✅ Successfully saved CoT Hallucination analysis to: {csv_path}")
    
    return df

# Execute the standalone analysis
df_cot = run_cot_hallucination_analysis()

Parsing raw outputs to detect Chain-of-Thought (CoT) induced hallucinations...

=== A5: CoT-Induced Hallucination Analysis ===
Measures how often the model generated the correct answer during reasoning, but outputted the wrong final answer.

                                            Model Dataset  Failed CoT Qs  CoT Hallucinations  CoT Hallucination Rate (%)
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL   SLAKE            479                 167                       34.86
                             google/gemma-3-4b-it  OK-VQA            413                 135                       32.69
                             google/gemma-3-4b-it   SLAKE            524                 180                       34.35
                             google/gemma-3-4b-it   VQAv2            406                  92                       22.66
                            google/medgemma-4b-it   SLAKE            281                  67                       23.84
                llava-hf/llava-v

In [7]:
# C2 — Failure Overlap Analysis

import os
import json
import glob
import pandas as pd
import itertools

def run_pairwise_overlap_analysis():
    judged_dir = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')
    # Focus specifically on SLAKE as requested
    judged_files = glob.glob(os.path.join(judged_dir, '*slake*_judged.jsonl'))
    
    # Nested dictionary to store exact question results: {model_name: {idx: is_correct}}
    model_results = {}
    
    print("Parsing SLAKE results for Pairwise Overlap Analysis...")
    
    for filepath in judged_files:
        filename = os.path.basename(filepath)
        model_name = filename.split('__')[0].replace('_', '/')
        
        if model_name not in model_results:
            model_results[model_name] = {}
            
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                idx = data.get('idx')
                judge_score = data.get('judge_score')
                
                # We need valid indices and judge scores to do a 1:1 comparison
                if idx is None or judge_score is None:
                    continue
                    
                # Semantic correctness baseline
                is_correct = True if judge_score >= 4 else False
                model_results[model_name][idx] = is_correct
                
    models = sorted(list(model_results.keys()))
    overlap_records = []
    
    # Generate all unique pairs of models
    for model_a, model_b in itertools.combinations(models, 2):
        # Ensure we are only comparing questions both models actually answered
        common_idxs = set(model_results[model_a].keys()) & set(model_results[model_b].keys())
        
        if not common_idxs:
            continue
            
        shared_wins = 0
        shared_failures = 0
        exclusive_a = 0
        exclusive_b = 0
        
        for idx in common_idxs:
            res_a = model_results[model_a][idx]
            res_b = model_results[model_b][idx]
            
            if res_a and res_b:
                shared_wins += 1
            elif not res_a and not res_b:
                shared_failures += 1
            elif res_a and not res_b:
                exclusive_a += 1
            elif not res_a and res_b:
                exclusive_b += 1
                
        overlap_records.append({
            'Model A': model_a,
            'Model B': model_b,
            'Common Qs': len(common_idxs),
            'Shared Wins': shared_wins,
            'Shared Failures': shared_failures,
            'Model A Exclusive Wins': exclusive_a,
            'Model B Exclusive Wins': exclusive_b
        })
        
    df = pd.DataFrame(overlap_records)
    
    # --- OUTPUT AND EXPORT ---
    output_dir = os.path.expanduser('~/vlm_benchmark/outputs/error_analysis/')
    os.makedirs(output_dir, exist_ok=True)
    
    csv_path = os.path.join(output_dir, 'pairwise_model_overlap.csv')
    df.to_csv(csv_path, index=False)
    
    print("\n=== C6: Pairwise Model Overlap (Success & Failure Concordance) ===")
    print("Analyzes whether models fail on the same difficult questions or have unique failure modes.\n")
    print(df.to_string(index=False))
    
    print(f"\n✅ Successfully saved Pairwise Overlap analysis to: {csv_path}")
    
    return df

# Execute the standalone analysis
df_overlap = run_pairwise_overlap_analysis()

Parsing SLAKE results for Pairwise Overlap Analysis...

=== C6: Pairwise Model Overlap (Success & Failure Concordance) ===
Analyzes whether models fail on the same difficult questions or have unique failure modes.

                                          Model A                             Model B  Common Qs  Shared Wins  Shared Failures  Model A Exclusive Wins  Model B Exclusive Wins
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL                google/gemma-3-4b-it       1061          502              308                     168                      83
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL               google/medgemma-4b-it       1061          623              232                      47                     159
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL   llava-hf/llava-v1.6-mistral-7b-hf       1061          432              291                     238                     100
FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL microsoft/llava-med-v1.5-mistral-7b